# Explainable Transformer for Postpartum Distress Detection

Fine-tunes a transformer (default: `mental/mental-roberta-base`, or swap in `dmis-lab/biobert-base-cased-v1.2`) on the 3-class distress dataset (Normal / Mild Distress / Emotionally Distressed), then explains predictions with SHAP.

**Run this on Google Colab** (Runtime -> Change runtime type -> GPU). It needs internet access to download pretrained weights, which a fully offline sandbox cannot do.

**Steps:**
1. Upload `processed_dataset.csv` (from the `data/` folder of the project) using the file upload cell below.
2. Run all cells.
3. Download the `transformer_model.zip` produced at the end.
4. Unzip it into your project's `models/transformer/` folder — the FastAPI app (`app/model_service.py`) will automatically detect and use it instead of the TF-IDF baseline.

In [ ]:
!pip install -q transformers datasets accelerate shap scikit-learn pandas

In [ ]:
from google.colab import files
print('Upload processed_dataset.csv')
uploaded = files.upload()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('processed_dataset.csv')
print(df['label_name'].value_counts())

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print('train:', len(train_df), 'test:', len(test_df))

In [ ]:
MODEL_NAME = "mental/mental-roberta-base"  # domain-adapted RoBERTa for mental-health text
# Alternative: "dmis-lab/biobert-base-cased-v1.2" (biomedical BERT)
# Alternative (lighter/faster if Colab GPU quota is tight): "distilbert-base-uncased"

NUM_LABELS = 3
LABEL_NAMES = {0: "Normal", 1: "Mild Distress", 2: "Emotionally Distressed"}

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

train_ds = Dataset.from_pandas(train_df[['text', 'label']].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[['text', 'label']].reset_index(drop=True))

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

train_ds = train_ds.rename_column('label', 'labels')
test_ds = test_ds.rename_column('label', 'labels')
train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
    }

args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    logging_steps=25,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
preds_output = trainer.predict(test_ds)
preds = np.argmax(preds_output.predictions, axis=-1)
labels = preds_output.label_ids
print(classification_report(labels, preds, target_names=[LABEL_NAMES[i] for i in range(3)]))

## Explainability with SHAP

Uses SHAP's `Explainer` with the HF `text-classification` pipeline to get per-word contribution scores for any prediction — this is what powers the 'why' explanation shown to the mother and doctor in the app.

In [ ]:
import shap
from transformers import pipeline

clf_pipeline = pipeline('text-classification', model=model, tokenizer=tokenizer,
                          top_k=None, device=0 if model.device.type == 'cuda' else -1)
explainer = shap.Explainer(clf_pipeline)

sample_texts = [
    "I feel so hopeless and empty, I don't think I can go on like this",
    "I'm tired but happy, my baby smiled at me today",
]
shap_values = explainer(sample_texts)
shap.plots.text(shap_values[0])

In [ ]:
def explain_prediction(text, top_k=8):
    sv = explainer([text])
    pred_idx = int(np.argmax(clf_pipeline(text)[0], axis=None)) if False else None
    # get predicted class index directly from the pipeline output
    scores = {d['label']: d['score'] for d in pipeline('text-classification', model=model, tokenizer=tokenizer, top_k=None)(text)[0]}
    pred_label = max(scores, key=scores.get)
    class_idx = int(pred_label.split('_')[-1]) if 'LABEL' in pred_label else list(LABEL_NAMES.values()).index(pred_label)
    word_contribs = list(zip(sv.data[0], sv.values[0][:, class_idx]))
    word_contribs.sort(key=lambda x: abs(x[1]), reverse=True)
    return {
        'label_name': LABEL_NAMES.get(class_idx, pred_label),
        'confidence': scores[pred_label],
        'top_words': [{'word': w.strip(), 'weight': float(v)} for w, v in word_contribs[:top_k] if w.strip()],
    }

explain_prediction("I feel like such a failure as a mother and I can't stop crying")

## Save and export the model

Downloads `transformer_model.zip` — unzip into your project as `models/transformer/`.

In [ ]:
SAVE_DIR = 'transformer'
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

import shutil
shutil.make_archive('transformer_model', 'zip', '.', SAVE_DIR)

from google.colab import files
files.download('transformer_model.zip')